In [1]:
import pandas as pd
import ast
import re
import random
import string
import warnings
warnings.filterwarnings('ignore')


In [2]:
random_state = 42

In [ ]:
all_data = pd.read_csv("../../data/nacc/training_data/training_data_grpo/train_summary.csv")
print("Length of all_data:", len(all_data))

Length of all_data: 43064


In [4]:
np_question_variants_one = [
    "Which of the following pathology is causing the patient's cognitive impairment based on the clinical information provided?",
    "Using the provided clinical information, determine the pathological cause of the patient's cognitive decline from the options listed below.",
    "Based on the clinical data, identify the neuropathology responsible for the patient's cognitive impairment by selecting from the given choices.",
    "From the options below, select the disease pathology causing cognitive impairment as indicated by the clinical information.",
    "Review the provided clinical information and choose the pathology underlying the patient's cognitive symptoms from the available options."
]

np_question_variants_mixed = [
    "Which of the following pathologies are causing the patient's cognitive impairment based on the clinical information provided?",
    "Using the provided clinical information, determine the pathological causes of the patient's cognitive decline from the options listed below.",
    "Based on the clinical data, identify the neuropathologies responsible for the patient's cognitive impairment by selecting from the given choices.",
    "From the options below, select the disease pathologies causing cognitive impairment as indicated by the clinical information.",
    "Review the provided clinical information and choose the pathologies underlying the patient's cognitive symptoms from the available options."
]

In [5]:
# Pull the bare answer texts into a list – easier to shuffle
NP_OPTION_TEXTS_ONE = [
    "Alzheimer's disease pathology (AD)",
    "Lewy body pathology (LBD)",
    "Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "No listed option is correct",
]

NP_OPTION_TEXTS_MIXED = [
    "Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)",
    "Alzheimer's disease pathology (AD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "Alzheimer's disease pathology (AD), Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "No listed option is correct",
]

def add_np_question(row, *, NP_OPTION_TEXTS, np_question_variants, seed=None):
    """
    Adds three new columns to each row:
    ─ question : one of the phrasing variants           (string)
    ─ options  : the shuffled, letter-prefixed choices  (multiline string)
    ─ ground_truth : the **letter** (A-H) of the correct choice (string)
    """
    rng = random.Random(seed if seed is not None else row.name)  # reproducible per-row if you like

    # 1️⃣ Choose a question at random
    row["question"] = rng.choice(np_question_variants)

    # 2️⃣ Shuffle the options and assign fresh letters
    shuffled = NP_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[: len(shuffled)])
    row["options"] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Work out which *text* is correct for this patient
    # combo = (row["NP_AD"], row["NP_LBD"], row["NP_FTD"])
    # correct_text = NP_ANSWER_MAP[combo]

    # 4️⃣ Convert that to the new letter after shuffling
    correct_letter = letters[shuffled.index(row['ground_truth_text'].replace(" only", ""))]
    row["ground_truth"] = correct_letter
    
    row['ground_truth_text'] = row['ground_truth_text'].replace(" only", "")

    return row

def split_np_cases(df):

    # Select cases based on the ground_truth_text column, not the options column
    one_pathology = df[df['ground_truth_text'].str.contains("only", na=False)].reset_index(drop=True)
    mixed_pathology = df[df['ground_truth_text'].str.contains("and", na=False)].reset_index(drop=True)
    no_pathology = df[df['ground_truth_text'] == "No listed option is correct"].reset_index(drop=True)

    # Split no_pathology cases between one_pathology and mixed_pathology
    half = len(no_pathology) // 2
    one_pathology = pd.concat([one_pathology, no_pathology.iloc[:half]]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    mixed_pathology = pd.concat([mixed_pathology, no_pathology.iloc[half:]]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    # Apply add_np_question with the correct option sets
    one_pathology = one_pathology.apply(add_np_question, NP_OPTION_TEXTS=NP_OPTION_TEXTS_ONE, np_question_variants=np_question_variants_one, axis=1)
    mixed_pathology = mixed_pathology.apply(add_np_question, NP_OPTION_TEXTS=NP_OPTION_TEXTS_MIXED, np_question_variants=np_question_variants_mixed, axis=1)
    
    assert len(set(one_pathology['ID']).intersection(set(mixed_pathology['ID']))) == 0
    
    return  pd.concat([one_pathology, mixed_pathology]).sample(frac=1, random_state=random_state).reset_index(drop=True)

In [6]:
def change_cog_cases(df_cog, df_mci, n=500):
    """
    Process cognitive status cases by creating balanced subsets and adding question variants.
    
    Args:
        df_cog: DataFrame containing cognitive status data
        df_mci: DataFrame containing MCI subtype data
    
    Returns:
        DataFrame with processed cognitive status cases including question variants
    """
    import json
    import random
    import string
    
    # Mapping for cognitive status
    mapping = {
        'Normal cognition': 'Normal Cognition (NC)',
        'MCI': 'Mild Cognitive Impairment (MCI)',
        'Dementia': 'Dementia (DE)'
    }
    
    def get_cog_stat(row):
        """Extract cognitive status from diagnosis summary"""
        try:
            diag_summary = json.loads(row['diag_summary'])
            row['COG'] = mapping[diag_summary['Clinician Diagnosis']['Cognitive status at UDS visit']]
        except (KeyError, json.JSONDecodeError, TypeError):
            # Handle cases where diag_summary might be missing or malformed
            row['COG'] = 'Unknown'
        return row
    
    # Combine cognitive status and MCI subtype data
    sub_df = pd.concat([df_cog, df_mci]).reset_index(drop=True)
    
    # Apply cognitive status mapping
    sub_df = sub_df.apply(get_cog_stat, axis=1)
    
    # Get indices for each COG group
    normal_idx = sub_df[sub_df['COG'] == 'Normal Cognition (NC)'].index
    mci_idx = sub_df[sub_df['COG'] == 'Mild Cognitive Impairment (MCI)'].index
    dementia_idx = sub_df[sub_df['COG'] == 'Dementia (DE)'].index
    
    # Calculate 50% sample sizes for each group
    n_normal_50 = int(0.5 * len(normal_idx))
    n_mci_50 = int(0.5 * len(mci_idx))
    n_dementia_50 = int(0.5 * len(dementia_idx))
    
    # Shuffle indices for each group to ensure randomness and reproducibility
    normal_idx_shuffled = normal_idx.to_series().sample(frac=1, random_state=random_state).tolist()
    mci_idx_shuffled = mci_idx.to_series().sample(frac=1, random_state=random_state).tolist()
    dementia_idx_shuffled = dementia_idx.to_series().sample(frac=1, random_state=random_state).tolist()
    
    # Create three balanced subsets with no overlap of IDs
    
    # 1st sub dataframe: 50% Normal cognition + 50% MCI
    sub_df1_normal_idx = normal_idx_shuffled[:n_normal_50]
    sub_df1_mci_idx = mci_idx_shuffled[:n_mci_50]
    sub_df1 = pd.concat([
        sub_df.loc[sub_df1_normal_idx],
        sub_df.loc[sub_df1_mci_idx]
    ])


    # 2nd sub dataframe: 50% MCI + 50% Dementia
    sub_df2_mci_idx = mci_idx_shuffled[n_mci_50:]
    sub_df2_dementia_idx = dementia_idx_shuffled[:n_dementia_50]
    sub_df2 = pd.concat([
        sub_df.loc[sub_df2_mci_idx],
        sub_df.loc[sub_df2_dementia_idx]
    ])


    # 3rd sub dataframe: 50% Normal cognition + 50% Dementia
    sub_df3_normal_idx = normal_idx_shuffled[n_normal_50:]
    sub_df3_dementia_idx = dementia_idx_shuffled[n_dementia_50:]
    sub_df3 = pd.concat([
        sub_df.loc[sub_df3_normal_idx],
        sub_df.loc[sub_df3_dementia_idx]
    ])
    
    # Reset indices and shuffle each sub dataframe
    sub_df1 = sub_df1.sample(frac=1, random_state=random_state).reset_index(drop=True)
    sub_df2 = sub_df2.sample(frac=1, random_state=random_state).reset_index(drop=True)
    sub_df3 = sub_df3.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # Verify no overlap in IDs
    assert set(sub_df1['ID']).isdisjoint(set(sub_df2['ID'])), "There are repeated IDs in sub_df1 and sub_df2"
    assert set(sub_df2['ID']).isdisjoint(set(sub_df3['ID'])), "There are repeated IDs in sub_df2 and sub_df3"
    assert set(sub_df1['ID']).isdisjoint(set(sub_df3['ID'])), "There are repeated IDs in sub_df1 and sub_df3"
    
    # Question variants for cognitive status identification
    cog_question_variants = [
        "Using the information provided, determine the patient's cognitive status.",
        "Identify the patient's cognitive status based on the given information.",
        "Based on the provided clinical details, classify the patient's cognitive status.",
        "From the information available, determine the correct cognitive status for the patient.",
        "Analyze the patient's information and identify their cognitive status."
    ]
    
    # Original answer texts
    COG_OPTION_TEXTS = [
        "Normal Cognition (NC)",
        "Mild Cognitive Impairment (MCI)",
        "Dementia (DE)"
    ]
    
    # Define which options to show for each subset
    SUBSET_OPTIONS = {
        'sub_df1': ["Normal Cognition (NC)", "Mild Cognitive Impairment (MCI)"],
        'sub_df2': ["Mild Cognitive Impairment (MCI)", "Dementia (DE)"],
        'sub_df3': ["Normal Cognition (NC)", "Dementia (DE)"]
    }
    
    def add_cog_question(row, subset_name, *, seed=None):
        """Add question variants and shuffled options for cognitive status questions"""
        rng = random.Random(seed if seed is not None else row.name)
        
        # 1️⃣ Random question phrasing
        row['question'] = rng.choice(cog_question_variants)
        
        # 2️⃣ Get the relevant options for this subset
        subset_options = SUBSET_OPTIONS.get(subset_name, COG_OPTION_TEXTS)
        
        # Shuffle the answer options
        shuffled = subset_options.copy()
        rng.shuffle(shuffled)
        letters = list(string.ascii_uppercase[:len(shuffled)])
        row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))
        
        # 3️⃣ Determine correct answer letter
        correct_text = row['COG']
        if correct_text in shuffled:
            row['ground_truth'] = letters[shuffled.index(correct_text)]
        else:
            row['ground_truth'] = None  # fallback for unrecognized codes
        
        row['ground_truth_text'] = correct_text
        
        return row
    
    # Apply question generation to each subset
    sub_df1 = sub_df1.apply(lambda row: add_cog_question(row, 'sub_df1'), axis=1).reset_index(drop=True)
    sub_df2 = sub_df2.apply(lambda row: add_cog_question(row, 'sub_df2'), axis=1).reset_index(drop=True)
    sub_df3 = sub_df3.apply(lambda row: add_cog_question(row, 'sub_df3'), axis=1).reset_index(drop=True)
    
    combined_cog_df = pd.concat([sub_df1, sub_df2, sub_df3]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    combined_cog_df['Q_TYPE'] = 'Cognitive status'
    combined_cog_df.drop(['COG'], inplace=True, axis=1)
    
    return combined_cog_df

In [7]:
def modify_data(df):
    np_data = split_np_cases(df[df['Q_TYPE'] == 'Neuropath'])
    cog_data = change_cog_cases(df[df['Q_TYPE'] == 'Cognitive status'], df[df['Q_TYPE'] == 'MCI subtype'])
    other_data = df[~df['Q_TYPE'].isin(['Cognitive status', 'MCI subtype', 'Neuropath'])]
    combined_data = pd.concat([np_data, cog_data, other_data]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    assert combined_data['ID'].is_unique, "combined_data has repeated IDs"
    
    return combined_data

In [8]:
modify_nacc_data = modify_data(all_data)

In [9]:
assert len(modify_nacc_data) == len(all_data)

In [10]:
all_data['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    20966
Primary etiology    11342
MCI subtype          5756
Amyloid PET          2000
Neuropath            2000
Amyloid CSF           800
DATSCAN               200
Name: count, dtype: int64

In [11]:
modify_nacc_data['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    26722
Primary etiology    11342
Neuropath            2000
Amyloid PET          2000
Amyloid CSF           800
DATSCAN               200
Name: count, dtype: int64

In [12]:
modify_nacc_data[modify_nacc_data['Q_TYPE'] == 'Neuropath']['question'].value_counts()

question
Which of the following pathology is causing the patient's cognitive impairment based on the clinical information provided?                           237
From the options below, select the disease pathology causing cognitive impairment as indicated by the clinical information.                          229
Review the provided clinical information and choose the pathology underlying the patient's cognitive symptoms from the available options.            222
Using the provided clinical information, determine the pathological cause of the patient's cognitive decline from the options listed below.          217
Based on the clinical data, identify the neuropathology responsible for the patient's cognitive impairment by selecting from the given choices.      206
Which of the following pathologies are causing the patient's cognitive impairment based on the clinical information provided?                        185
Using the provided clinical information, determine the pathological cause

In [ ]:
modify_nacc_data.to_csv("../../data/nacc/training_data/training_data_grpo_modified_cog_np/train_summary.csv", index=False)

In [14]:
for qtype in modify_nacc_data['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(modify_nacc_data[modify_nacc_data['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


Q_TYPE: Cognitive status
ground_truth_text
Normal Cognition (NC)              13468
Dementia (DE)                       7283
Mild Cognitive Impairment (MCI)     5971
Name: count, dtype: int64

Q_TYPE: Neuropath
ground_truth_text
Alzheimer's disease pathology (AD)                                                                                                                   633
Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)                                                                                     585
Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                                                      299
No listed option is correct                                                                                                                          165
Lewy body pathology (LBD)                                                                                                                             97
Alzhei

In [ ]:
train = pd.read_csv(f"../../data/nacc/train.csv")

In [ ]:
new_train = train.drop(['patient_summary', 'diag_summary', 'COHORT'], axis=1, inplace=False)
common_cols = [col for col in modify_nacc_data.columns if col in new_train.columns]
merged_df = pd.merge(modify_nacc_data, new_train, on=common_cols, how='inner')


In [ ]:
merged_df.to_csv(f"../../data/nacc/training_data/training_data_grpo_modified_cog_np/train_with_questions.csv", index=False)